# 🏪 Thực hành Sắp xếp và biến đổi dữ liệu

Trong notebook này, chúng ta sẽ luyện tập các kỹ thuật:
- Hierarchical Indexing
- Combining & Merging
- Concatenation
- Reshaping & Pivoting

Dữ liệu sử dụng: [Walmart Store Sales Forecasting](https://www.kaggle.com/competitions/walmart-recruiting-store-sales-forecasting)

---

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Đọc dữ liệu (ưu tiên data/; fallback sang data/walmart/)
base = Path('data')
train_path = base / 'train.csv'
features_path = base / 'features.csv'
stores_path = base / 'stores.csv'

if not train_path.exists():
    train_path = base / 'walmart' / 'train.csv'
if not features_path.exists():
    features_path = base / 'walmart' / 'features.csv'
if not stores_path.exists():
    stores_path = base / 'walmart' / 'stores.csv'

train = pd.read_csv(train_path)
features = pd.read_csv(features_path)
stores = pd.read_csv(stores_path)

train['Date'] = pd.to_datetime(train['Date'])
features['Date'] = pd.to_datetime(features['Date'])

print('Loaded:', train_path, features_path, stores_path)
train.head()

Loaded: data/train.csv data/features.csv data/stores.csv


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


# Phần 1: Chỉ số phân cấp
**Ngữ cảnh:** Dữ liệu doanh số theo `Store` và `Dept`. Nếu chỉ dùng index 1 cấp thì khó truy vấn chi tiết. MultiIndex sẽ hữu ích.

---

**Bài 1:** Tạo MultiIndex từ hai cột `Store` và `Dept`, sau đó hiển thị dữ liệu của `Store = 1, Dept = 1`. 

In [3]:
# Tạo MultiIndex từ Store và Dept
train_mi = train.set_index(['Store', 'Dept']).sort_index()

# Hiển thị dữ liệu của Store=1, Dept=1
store1_dept1 = train_mi.loc[(1, 1)]
print("Dữ liệu Store=1, Dept=1:")
print(store1_dept1.head())

Dữ liệu Store=1, Dept=1:
                 Date  Weekly_Sales  IsHoliday
Store Dept                                    
1     1    2010-02-05      24924.50      False
      1    2010-02-12      46039.49       True
      1    2010-02-19      41595.55      False
      1    2010-02-26      19403.54      False
      1    2010-03-05      21827.90      False


**Bài 2:** Lấy toàn bộ dữ liệu của `Store = 2` (tất cả Dept).

In [4]:
# Lấy toàn bộ dữ liệu của Store=2 (tất cả Dept)
store2_all = train_mi.loc[2]
print("Dữ liệu Store=2 (tất cả Dept):")
print(store2_all.head())

Dữ liệu Store=2 (tất cả Dept):
           Date  Weekly_Sales  IsHoliday
Dept                                    
1    2010-02-05      35034.06      False
1    2010-02-12      60483.70       True
1    2010-02-19      58221.52      False
1    2010-02-26      25962.32      False
1    2010-03-05      27372.05      False


**Bài 3:** Lấy dữ liệu của tất cả các `Dept = 5` trên toàn bộ các `Store`.

In [5]:
# Lấy dữ liệu Dept=5 trên toàn bộ các Store
dept5_all = train_mi.xs(5, level='Dept')
print("Dữ liệu Dept=5 trên toàn bộ Store:")
print(dept5_all.head())

Dữ liệu Dept=5 trên toàn bộ Store:
            Date  Weekly_Sales  IsHoliday
Store                                    
1     2010-02-05      32229.38      False
1     2010-02-12      29620.81       True
1     2010-02-19      26468.27      False
1     2010-02-26      24101.89      False
1     2010-03-05      23082.14      False


**Bài 4:** Tính doanh số trung bình cho từng `Dept` trong mỗi `Store` bằng `groupby` rồi chuyển kết quả thành MultiIndex.

In [6]:
# Tính doanh số trung bình theo từng Dept trong mỗi Store
avg_sales_store_dept = (
    train.groupby(['Store', 'Dept'])['Weekly_Sales']
    .mean()
    .to_frame('Avg_Weekly_Sales')
)

print("Doanh số trung bình theo Store-Dept (MultiIndex):")
print(avg_sales_store_dept.head(10))

Doanh số trung bình theo Store-Dept (MultiIndex):
            Avg_Weekly_Sales
Store Dept                  
1     1         22513.322937
      2         46102.090420
      3         13150.478042
      4         36964.154476
      5         24257.941119
      6          4801.780140
      7         24566.487413
      8         35718.257622
      9         28062.052238
      10        31033.386364


**Bài 5:** Sau khi tạo MultiIndex, thử dùng `.xs()` để truy cập nhanh một cấp chỉ số (ví dụ: Dept = 10 của tất cả các Store).

In [7]:
# Dùng xs để lấy Dept=10 của tất cả Store
dept10_all_stores = train_mi.xs(10, level='Dept')
print("Dữ liệu Dept=10 của tất cả Store:")
print(dept10_all_stores.head())

Dữ liệu Dept=10 của tất cả Store:
            Date  Weekly_Sales  IsHoliday
Store                                    
1     2010-02-05      30721.50      False
1     2010-02-12      31494.77       True
1     2010-02-19      29634.13      False
1     2010-02-26      27921.96      False
1     2010-03-05      33299.27      False


# Phần 2: Kết hợp các tập dữ liệu
**Ngữ cảnh:** Dữ liệu doanh số (`train.csv`) cần ghép với thông tin kinh tế khác (`features.csv`) để thấy được các yếu tố khác ảnh hưởng tới doanh số như thế nào.

---

**Bài 1:** Merge `train` với `features` dựa trên `Store` và `Date` (left join).

In [8]:
# Merge train với features theo Store và Date (left join)
train_features = train.merge(features, on=['Store', 'Date'], how='left', suffixes=('', '_feat'))

print("Kích thước sau merge train-features:", train_features.shape)
print(train_features.head())

Kích thước sau merge train-features: (421570, 15)
   Store  Dept       Date  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  \
0      1     1 2010-02-05      24924.50      False        42.31       2.572   
1      1     1 2010-02-12      46039.49       True        38.51       2.548   
2      1     1 2010-02-19      41595.55      False        39.93       2.514   
3      1     1 2010-02-26      19403.54      False        46.63       2.561   
4      1     1 2010-03-05      21827.90      False        46.50       2.625   

   MarkDown1  MarkDown2  MarkDown3  MarkDown4  MarkDown5         CPI  \
0        NaN        NaN        NaN        NaN        NaN  211.096358   
1        NaN        NaN        NaN        NaN        NaN  211.242170   
2        NaN        NaN        NaN        NaN        NaN  211.289143   
3        NaN        NaN        NaN        NaN        NaN  211.319643   
4        NaN        NaN        NaN        NaN        NaN  211.350143   

   Unemployment  IsHoliday_feat  
0       

**Bài 2:** Kiểm tra có bao nhiêu bản ghi trong `train` không tìm thấy trong `features`.

In [9]:
# Các cột thuộc features (trừ key)
feature_cols = [c for c in features.columns if c not in ['Store', 'Date']]

# Đếm bản ghi không match features: toàn bộ cột feature bị NaN
missing_in_features = train_features[feature_cols].isna().all(axis=1).sum()
print("Số bản ghi train không tìm thấy trong features:", missing_in_features)

Số bản ghi train không tìm thấy trong features: 0


**Bài 3:** Thêm dữ liệu `stores.csv` để có thêm cột `Type` và `Size` cho mỗi `Store`.

In [10]:
# Merge thêm stores để có Type và Size
full_data = train_features.merge(stores, on='Store', how='left')

print("Kích thước sau khi merge đủ 3 bảng:", full_data.shape)
print(full_data[['Store', 'Dept', 'Date', 'Weekly_Sales', 'Type', 'Size']].head())

Kích thước sau khi merge đủ 3 bảng: (421570, 17)
   Store  Dept       Date  Weekly_Sales Type    Size
0      1     1 2010-02-05      24924.50    A  151315
1      1     1 2010-02-12      46039.49    A  151315
2      1     1 2010-02-19      41595.55    A  151315
3      1     1 2010-02-26      19403.54    A  151315
4      1     1 2010-03-05      21827.90    A  151315


**Bài 4:** Tìm `Store` nào có doanh số trung bình cao nhất khi merge đủ cả 3 bảng (`train`, `features`, `stores`).

In [11]:
# Doanh số trung bình theo Store
store_avg_sales = full_data.groupby('Store')['Weekly_Sales'].mean().sort_values(ascending=False)

best_store = store_avg_sales.index[0]
best_value = store_avg_sales.iloc[0]

print(f"Store có doanh số trung bình cao nhất: {best_store}")
print(f"Doanh số trung bình: {best_value:,.2f}")

Store có doanh số trung bình cao nhất: 20
Doanh số trung bình: 29,508.30


**Bài 5:** Sau khi merge, tính doanh số trung bình của các `Store` theo từng loại `Type` (A/B/C).

In [12]:
# Doanh số trung bình theo Type (A/B/C)
type_avg_sales = full_data.groupby('Type')['Weekly_Sales'].mean().sort_values(ascending=False)

print("Doanh số trung bình theo loại Store:")
print(type_avg_sales)

Doanh số trung bình theo loại Store:
Type
A    20099.568043
B    12237.075977
C     9519.532538
Name: Weekly_Sales, dtype: float64


**Bài 6:** Chia `train` thành 3 DataFrame theo năm (2010, 2011, 2012).

In [13]:
# Chia train thành 3 DataFrame theo năm
train_2010 = train[train['Date'].dt.year == 2010].copy()
train_2011 = train[train['Date'].dt.year == 2011].copy()
train_2012 = train[train['Date'].dt.year == 2012].copy()

print("Số dòng theo năm:")
print("2010:", len(train_2010))
print("2011:", len(train_2011))
print("2012:", len(train_2012))

Số dòng theo năm:
2010: 140679
2011: 153453
2012: 127438


**Bài 7:** Ghép chúng lại thành một DataFrame duy nhất bằng `pd.concat`.

In [14]:
# Ghép 3 DataFrame lại bằng concat theo hàng
train_concat = pd.concat([train_2010, train_2011, train_2012], axis=0, ignore_index=True)

print("Kích thước sau concat theo hàng:", train_concat.shape)
print(train_concat.head())

Kích thước sau concat theo hàng: (421570, 5)
   Store  Dept       Date  Weekly_Sales  IsHoliday
0      1     1 2010-02-05      24924.50      False
1      1     1 2010-02-12      46039.49       True
2      1     1 2010-02-19      41595.55      False
3      1     1 2010-02-26      19403.54      False
4      1     1 2010-03-05      21827.90      False


**Bài 8:** Thử ghép theo **cột** (`axis=1`) và quan sát sự khác biệt.

In [15]:
# Concat theo cột (axis=1)
concat_axis1 = pd.concat(
    [
        train_2010.reset_index(drop=True).add_prefix('2010_'),
        train_2011.reset_index(drop=True).add_prefix('2011_'),
        train_2012.reset_index(drop=True).add_prefix('2012_')
    ],
    axis=1
)

print("Kích thước concat theo cột:", concat_axis1.shape)
print(concat_axis1.head())

# Khác biệt: axis=0 nối thêm dòng; axis=1 nối thêm cột theo chỉ mục dòng.

Kích thước concat theo cột: (153453, 15)
   2010_Store  2010_Dept  2010_Date  2010_Weekly_Sales 2010_IsHoliday  \
0         1.0        1.0 2010-02-05           24924.50          False   
1         1.0        1.0 2010-02-12           46039.49           True   
2         1.0        1.0 2010-02-19           41595.55          False   
3         1.0        1.0 2010-02-26           19403.54          False   
4         1.0        1.0 2010-03-05           21827.90          False   

   2011_Store  2011_Dept  2011_Date  2011_Weekly_Sales  2011_IsHoliday  \
0           1          1 2011-01-07           15984.24           False   
1           1          1 2011-01-14           17359.70           False   
2           1          1 2011-01-21           17341.47           False   
3           1          1 2011-01-28           18461.18           False   
4           1          1 2011-02-04           21665.76           False   

   2012_Store  2012_Dept  2012_Date  2012_Weekly_Sales 2012_IsHoliday  
0  

**Bài 9:** Gán keys = `['2010','2011','2012']` khi concat để tạo MultiIndex ở cấp trên.

In [16]:
# Concat có keys để tạo MultiIndex ở cấp trên
concat_with_keys = pd.concat(
    [train_2010, train_2011, train_2012],
    keys=['2010', '2011', '2012'],
    names=['YearBlock']
)

print("Dữ liệu concat với keys (MultiIndex):")
print(concat_with_keys.head())
print("\nTên index levels:", concat_with_keys.index.names)

Dữ liệu concat với keys (MultiIndex):
             Store  Dept       Date  Weekly_Sales  IsHoliday
YearBlock                                                   
2010      0      1     1 2010-02-05      24924.50      False
          1      1     1 2010-02-12      46039.49       True
          2      1     1 2010-02-19      41595.55      False
          3      1     1 2010-02-26      19403.54      False
          4      1     1 2010-03-05      21827.90      False

Tên index levels: ['YearBlock', None]


**Bài 10:** So sánh số bản ghi trước và sau khi concat để kiểm tra tính toàn vẹn dữ liệu.

In [17]:
# So sánh số bản ghi trước và sau concat
before_concat = len(train_2010) + len(train_2011) + len(train_2012)
after_concat = len(train_concat)

print("Số bản ghi trước concat:", before_concat)
print("Số bản ghi sau concat:", after_concat)
print("Toàn vẹn dữ liệu:", before_concat == after_concat)

Số bản ghi trước concat: 421570
Số bản ghi sau concat: 421570
Toàn vẹn dữ liệu: True


# Phần 3: Tổ chức và Cấu trúc lại dữ liệu
**Ngữ cảnh:** Ban lãnh đạo cần báo cáo doanh số dễ đọc hơn, thay vì dữ liệu dạng long.

---

**Bài 1:** Dùng `pivot_table` để tính doanh số trung bình theo `Store` (hàng) và `Dept` (cột).

In [18]:
# Pivot table: doanh số trung bình theo Store (hàng) và Dept (cột)
pivot_store_dept = train.pivot_table(
    values='Weekly_Sales',
    index='Store',
    columns='Dept',
    aggfunc='mean'
)

print("Pivot doanh số trung bình theo Store x Dept:")
print(pivot_store_dept.head())

Pivot doanh số trung bình theo Store x Dept:
Dept             1             2             3             4             5   \
Store                                                                         
1      22513.322937  46102.090420  13150.478042  36964.154476  24257.941119   
2      30777.980769  65912.922517  17476.563357  45607.666573  30555.315315   
3       7328.621049  16841.775664   5509.300769   8434.186503  11695.366573   
4      36979.940070  93639.315385  19012.491678  56603.400140  45668.406783   
5       9774.553077  12317.953287   4101.085175   9860.806783   6699.202238   

Dept            6             7             8             9             10  \
Store                                                                        
1      4801.780140  24566.487413  35718.257622  28062.052238  31033.386364   
2      6808.382517  40477.837063  58707.369441  34375.864476  38845.854476   
3      2012.411818  10044.341608   8310.254196   9062.007692  10871.944126   
4      8241

**Bài 2:** Lấy doanh số trung bình theo từng `Store` theo từng năm (sử dụng `Date`).

In [19]:
# Tạo cột Year từ Date rồi pivot theo Store x Year
train_with_year = train.copy()
train_with_year['Year'] = train_with_year['Date'].dt.year

pivot_store_year = train_with_year.pivot_table(
    values='Weekly_Sales',
    index='Store',
    columns='Year',
    aggfunc='mean'
)

print("Doanh số trung bình theo Store và năm:")
print(pivot_store_year.head())

Doanh số trung bình theo Store và năm:
Year           2010          2011          2012
Store                                          
1      21283.424920  21718.174673  22179.531063
2      27794.009390  26443.518750  26451.377920
3       6178.450560   6346.608710   6621.763226
4      27709.374692  29831.442892  29974.536103
5       4951.946185   4982.099214   5253.555109


**Bài 3:** Dùng `stack` và `unstack` để biến đổi bảng pivot thành long rồi wide trở lại.

In [20]:
# Stack: wide -> long
long_from_pivot = pivot_store_dept.stack(dropna=False).reset_index(name='Avg_Weekly_Sales')

# Unstack/pivot lại: long -> wide
wide_again = long_from_pivot.pivot(index='Store', columns='Dept', values='Avg_Weekly_Sales')

print("Dữ liệu sau stack (long):")
print(long_from_pivot.head())
print("\nDữ liệu sau unstack/pivot (wide lại):")
print(wide_again.head())

Dữ liệu sau stack (long):
   Store  Dept  Avg_Weekly_Sales
0      1     1      22513.322937
1      1     2      46102.090420
2      1     3      13150.478042
3      1     4      36964.154476
4      1     5      24257.941119

Dữ liệu sau unstack/pivot (wide lại):
Dept             1             2             3             4             5   \
Store                                                                         
1      22513.322937  46102.090420  13150.478042  36964.154476  24257.941119   
2      30777.980769  65912.922517  17476.563357  45607.666573  30555.315315   
3       7328.621049  16841.775664   5509.300769   8434.186503  11695.366573   
4      36979.940070  93639.315385  19012.491678  56603.400140  45668.406783   
5       9774.553077  12317.953287   4101.085175   9860.806783   6699.202238   

Dept            6             7             8             9             10  \
Store                                                                        
1      4801.780140  24566.4

/var/folders/8x/cphm0p8j0h16x_zm6w8f2nww0000gn/T/ipykernel_35978/3481188297.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  long_from_pivot = pivot_store_dept.stack(dropna=False).reset_index(name='Avg_Weekly_Sales')


**Bài 4:** Tạo pivot table để xem doanh số trung bình theo `Type` của Store và theo từng năm.

In [21]:
# Merge train với stores để có Type
train_store = train.merge(stores[['Store', 'Type']], on='Store', how='left')
train_store['Year'] = train_store['Date'].dt.year

# Pivot theo Type và Year
pivot_type_year = train_store.pivot_table(
    values='Weekly_Sales',
    index='Type',
    columns='Year',
    aggfunc='mean'
)

print("Doanh số trung bình theo Type và năm:")
print(pivot_type_year)

Doanh số trung bình theo Type và năm:
Year          2010          2011          2012
Type                                          
A     20327.226574  20111.476693  19832.335496
B     12627.241607  12177.025551  11877.700600
C      9571.743733   9402.674708   9602.105782
